# 03 - Data Quality Checks

Fail the notebook when a critical uniqueness, null, range, or referential-integrity rule is violated.


In [ ]:
from pyspark.sql import functions as F

customers=spark.table("silver_customers")
products=spark.table("silver_products")
sales=spark.table("silver_sales")
checks=[]

def check(name, passed, details=""):
    checks.append((name, bool(passed), details))

check("Unique customer_id", customers.count()==customers.select("customer_id").distinct().count())
check("Unique product_id", products.count()==products.select("product_id").distinct().count())
check("Unique order_id", sales.count()==sales.select("order_id").distinct().count())
check("No null order_date", sales.filter(F.col("order_date").isNull()).count()==0)
check("Non-negative sales", sales.filter(F.col("sales_amount")<0).count()==0)

oc=sales.join(customers,"customer_id","left_anti").count()
op=sales.join(products,"product_id","left_anti").count()
check("No orphan customer keys", oc==0, f"orphans={oc}")
check("No orphan product keys", op==0, f"orphans={op}")

for name, passed, details in checks:
    print(("PASS" if passed else "FAIL"), "-", name, details)

failed=[x for x in checks if not x[1]]
if failed:
    raise Exception(f"Data quality failed: {failed}")
